# 1. Imports

In [16]:
# Para analytics
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

# Para conexão com AWS
import awswrangler as wr
import boto3
import subprocess
import json

# Setup da conexão com o Athena

Durante a aula, o professor sugeriu que, no lugar da abordagem de usar um subprocesso, as informações do output fossem passadas por linha de comando. Acabei fazendo com subprocesso no notebook por não fazer sentido ter argumentos de terminal nesse tipo de arquivo/ambiente.

In [ ]:
pasta_terraform = "terraform_config/"

# Executa o comando terraform para capturar os outputs em formato JSON
resultado_tf = subprocess.run(
    ["terraform", "output", "-json"],
    cwd=pasta_terraform,
    capture_output=True,
    text=True
)

# Transforma a string JSON de saida em um dicionario Python
outputs = json.loads(resultado_tf.stdout)

# Pega os nomes dos recursos na AWS a partir do terraform
GLUE_DATABASE = outputs["glue_database_name"]["value"]
ATHENA_WORKGROUP = outputs["athena_workgroup_name"]["value"]

Essa célula testa a conexão e já faz o teste especificado na sessão "4.2 - Consulta exploratória em dimensão"

In [ ]:
session = boto3.Session(region_name="us-east-1")

query_teste = """
    SELECT
        product_id,
        product_name,
        product_line,
        product_vendor
    FROM dim_products
    LIMIT 20
"""

query_kwargs = {
    "database": GLUE_DATABASE,
    "boto3_session": session,
    "workgroup": ATHENA_WORKGROUP
}

df_produtos = wr.athena.read_sql_query(
    sql = query_teste,
    **query_kwargs
)

df_produtos.head()

,product_id,product_name,product_line,product_vendor
0,S18_2795,1928 Mercedes-Benz SSK,Vintage Cars,Gearbox Collectibles
1,S18_3232,1992 Ferrari 360 Spider red,Classic Cars,Unimax Art Galleries
2,S18_3259,Collectable Wooden Train,Trains,Carousel DieCast Legends
3,S18_4409,1932 Alfa Romeo 8C2300 Spider Sport,Vintage Cars,Exoto Designs
4,S18_1749,1917 Grand Touring Sedan,Vintage Cars,Welly Diecast Productions


# Análises

## Vendas totais por país

In [ ]:
query_vendas_pais = """
SELECT
    dim_countries.country,
    SUM(fact_orders.sales_amount) AS total_sales
FROM fact_orders
JOIN dim_countries ON fact_orders.country_key = dim_countries.country_key
GROUP BY dim_countries.country
ORDER BY total_sales DESC
LIMIT 10
"""

df_vendas = wr.athena.read_sql_query(
    sql = query_vendas_pais,
    **query_kwargs
)

df_vendas

,country,total_sales
0,USA,3273280.05
1,Spain,1099389.09
2,France,1007374.02
3,Australia,562582.59
4,New Zealand,476847.01
5,UK,436947.44
6,Italy,360616.81
7,Finland,295149.35
8,Singapore,263997.78
9,Denmark,218994.92


### Detalhes por data, linha de produto, produto e país

In [15]:
query_detalhes = """
SELECT
    dim_dates.full_date,
    dim_products.product_line,
    dim_products.product_name,
    dim_countries.country,
    SUM(fact_orders.sales_amount) AS total_sales
FROM fact_orders
JOIN dim_products ON fact_orders.product_id = dim_products.product_id
JOIN dim_countries ON fact_orders.country_key = dim_countries.country_key
JOIN dim_dates ON fact_orders.order_date_key = dim_dates.date_key
GROUP BY
    dim_dates.full_date,
    dim_products.product_line,
    dim_products.product_name,
    dim_countries.country
"""

df_detalhes = wr.athena.read_sql_query(
    sql = query_detalhes,
    **query_kwargs
)
df_detalhes["full_date"] = pd.to_datetime(df_detalhes["full_date"])
df_detalhes

,full_date,product_line,product_name,country,total_sales
0,2004-03-15,Classic Cars,1998 Chrysler Plymouth Prowler,Germany,7541.59
1,2003-10-04,Classic Cars,1969 Dodge Charger,Germany,2198.34
2,2004-11-05,Trucks and Buses,1940s Ford truck,Germany,4552.42
3,2004-11-09,Vintage Cars,1940 Ford Delivery Sedan,Sweden,1626.80
4,2003-03-24,Vintage Cars,1913 Ford Model T Speedster,Sweden,1957.30
...,...,...,...,...,...
2991,2004-12-16,Vintage Cars,1941 Chevrolet Special Deluxe Cabriolet,New Zealand,4065.60
2992,2004-12-16,Ships,The Queen Mary,New Zealand,2863.16
2993,2004-12-16,Vintage Cars,1912 Ford Model T Delivery Wagon,New Zealand,2549.16
2994,2004-12-16,Trains,Collectable Wooden Train,New Zealand,2748.91


## Dashboard

In [20]:
data_min = df_detalhes["full_date"].min().date()
data_max = df_detalhes["full_date"].max().date()

# Listas com as opções que vão para os seletores do dashboard
paises = ["Todos"] + sorted(df_detalhes["country"].dropna().unique().tolist())
linhas_produto = ["Todos"] + sorted(df_detalhes["product_line"].dropna().unique().tolist())

# Declaracao dos Widgets que farão parte do dashboard
data_inicio = widgets.DatePicker(description="Data Inicial:", value=data_min)
data_fim = widgets.DatePicker(description="Data Final:", value=data_max)
dropdown_pais = widgets.Dropdown(options=paises, value="Todos", description="Pais:")
dropdown_linha = widgets.Dropdown(options=linhas_produto, value="Todos", description="Linha:")
slider_topn = widgets.IntSlider(value=5, min=1, max=10, description="Top N:")

# Essa função usa os inputs obtidos pelos widgets para gerar o gráfico de top N
def atualizar_dashboard(pais, linha, data_ini, data_f, top_n):
    df_filtrado = df_detalhes.copy()
    
    ts_ini = pd.to_datetime(data_ini)
    ts_f = pd.to_datetime(data_f)
    df_filtrado = df_filtrado[(df_filtrado["full_date"] >= ts_ini) & (df_filtrado["full_date"] <= ts_f)]
    
    # Aplica os filtros dos widgets
    if pais != "Todos":
        df_filtrado = df_filtrado[df_filtrado["country"] == pais]
        
    if linha != "Todos":
        df_filtrado = df_filtrado[df_filtrado["product_line"] == linha]
        
    # Caso de borda em que não há dados de venda
    if df_filtrado.empty:
        print("Nenhum dado encontrado para os filtros selecionados no intervalo de datas.")
        return

    df_agrupado = df_filtrado.groupby("product_name", as_index=False)["total_sales"].sum()
    df_top_n = df_agrupado.sort_values(by="total_sales", ascending=False).head(top_n)
    
    # Configuração do gráfico com o seaborn
    plt.figure(figsize=(10, 6))
    sns.barplot(data=df_top_n, x="total_sales", y="product_name", hue="product_name", palette="viridis")
    
    plt.title(f"Top {top_n} Produtos Vendidos")
    plt.xlabel("Vendas Totais ($)")
    plt.ylabel("Nome do Produto")    
    plt.tight_layout()
    plt.show()

# Organiza os botoes
ui_filtros = widgets.VBox([
    widgets.HBox([data_inicio, data_fim]),
    widgets.HBox([dropdown_pais, dropdown_linha]),
    slider_topn
])

# Conecta os widgets com a funcao de plotagem
out = widgets.interactive_output(atualizar_dashboard, {
    "pais": dropdown_pais,
    "linha": dropdown_linha,
    "data_ini": data_inicio,
    "data_f": data_fim,
    "top_n": slider_topn
})

# Renderiza tudo na celula do Jupyter
display(ui_filtros, out)

Output()